In [18]:
# ⚙️ Global Config & Services (using centralized modules)

import sys
from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    validate_api_keys,
    print_config_summary,
    get_text_embeddings
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


Working directory: c:\Users\viraj\Zuu\Ledger_mind
✅ Config loaded:
  LLM: groq / llama-3.1-8b-instant
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


In [19]:
# Initialize LLM using factory from llm_services
llm = get_llm(config)
print(f"LLM initialized: {config['llm_provider']} / {config['llm_model']}")

# Verify API key with test completion
print("\n🔍 Testing API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"API key verified: {test_msg[:50]}")
except Exception as e:
    print(f" API key test failed: {e}")
    print("Please check your .env file and API key configuration.")


LLM initialized: groq / llama-3.1-8b-instant

🔍 Testing API connection...
API key verified: API working!


In [20]:
# Load documents using the utility module
from src.utils.document_loader import load_pdf_documents

# Load all PDFs from data directory
pdf_dir = Path(config["data_root"])
documents = load_pdf_documents(pdf_dir)

print(f"\n✅ Loaded {len(documents)} documents total")

✓ Loaded annual_report_2024.pdf: 640,073 chars from 142 pages

✅ Loaded 1 documents total


### Inspect Document Data

In [21]:
# Inspect loaded documents
import re

def inspect_document(doc, sample_lines=10):
    """Display comprehensive document statistics and samples."""
    content = doc['content']
    lines = content.split('\n')
    
    print(f"\n{'='*70}")
    print(f"📄 Document: {doc['source']}")
    print(f"{'='*70}")
    
    # Basic stats
    print(f"\n📊 Statistics:")
    print(f"  Total characters: {len(content):,}")
    print(f"  Total lines: {len(lines):,}")
    print(f"  Total words: {len(content.split()):,}")
    print(f"  Avg line length: {len(content) / max(len(lines), 1):.1f} chars")
    
    # Non-empty lines
    non_empty_lines = [line for line in lines if line.strip()]
    print(f"  Non-empty lines: {len(non_empty_lines):,}")
    
    # Sample from beginning
    print(f"\n📖 First {sample_lines} lines:")
    print("-" * 70)
    for i, line in enumerate(lines[:sample_lines], 1):
        display_line = line[:100] + "..." if len(line) > 100 else line
        print(f"  {i:3d}: {display_line}")
    
    # Sample from middle
    mid_point = len(lines) // 2
    print(f"\n📖 Sample from middle (around line {mid_point}):")
    print("-" * 70)
    for i, line in enumerate(lines[mid_point:mid_point+sample_lines], mid_point+1):
        display_line = line[:100] + "..." if len(line) > 100 else line
        print(f"  {i:3d}: {display_line}")
    
    # Sample from end
    print(f"\n📖 Last {sample_lines} lines:")
    print("-" * 70)
    start_idx = max(0, len(lines) - sample_lines)
    for i, line in enumerate(lines[-sample_lines:], start_idx+1):
        display_line = line[:100] + "..." if len(line) > 100 else line
        print(f"  {i:3d}: {display_line}")
    
    # Detect potential headers/footers
    print(f"\n🔍 Potential Headers/Footers:")
    print("-" * 70)
    
    # Look for page numbers
    page_num_pattern = r'^\s*(?:page\s*)?(\d+)(?:\s+of\s+\d+)?\s*$'
    page_nums = [line.strip() for line in lines if re.match(page_num_pattern, line.strip(), re.IGNORECASE)]
    if page_nums:
        print(f"  Page numbers found: {len(page_nums)} occurrences")
        print(f"    Examples: {', '.join(page_nums[:5])}")
    
    # Look for dates
    date_pattern = r'\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{4}'
    dates = [line.strip() for line in lines if re.match(f'^{date_pattern}$', line.strip(), re.IGNORECASE)]
    if dates:
        print(f"  Date footers found: {len(dates)} occurrences")
        print(f"    Examples: {', '.join(dates[:5])}")
    
    # Look for very short lines (potential artifacts)
    very_short = [line.strip() for line in lines if 0 < len(line.strip()) < 3]
    if very_short:
        print(f"  Very short lines (1-2 chars): {len(very_short)} occurrences")
        print(f"    Examples: {', '.join(very_short[:10])}")
    
    # Look for URLs
    url_pattern = r'https?://|www\.'
    urls = [line.strip() for line in lines if re.search(url_pattern, line.strip(), re.IGNORECASE)]
    if urls:
        print(f"  URLs found: {len(urls)} occurrences")
        print(f"    Examples: {urls[0][:60] if urls else ''}")
    
    print()

# Inspect all documents
for doc in documents:
    inspect_document(doc, sample_lines=10)


📄 Document: annual_report_2024.pdf

📊 Statistics:
  Total characters: 640,073
  Total lines: 7,255
  Total words: 98,684
  Avg line length: 88.2 chars
  Non-empty lines: 6,846

📖 First 10 lines:
----------------------------------------------------------------------
    1: On Our Way
    2: 2024 ANNUAL REPORT
    3: Uber’s Mission
    4: We reimagine the way the world moves for the better
    5: We are Uber. The go-getters. The kind of people who are relentless about our  
    6: mission to help people go anywhere and get anything and earn their way. 
    7: Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
    8: what gets us out of bed each morning. It pushes us to constantly reimagine 
    9: how we can move better. For you. For all the places you want to go. For all the 
   10: things you want to get. For all the ways you want to earn. Across the entire 

📖 Sample from middle (around line 3627):
---------------------------------------------------------

### Clean Data

In [22]:
# Clean documents using the utility module
from src.utils.document_loader import clean_documents, save_cleaned_text

# Clean documents (remove headers/footers)
cleaned_docs = clean_documents(documents, remove_footers=True)

# Save cleaned text files
output_dir = Path(config.get("data_root", "data")) / "cleaned"
save_cleaned_text(cleaned_docs, output_dir)

print(f"\n✅ Cleaned and saved {len(cleaned_docs)} documents")

✓ Cleaned annual_report_2024.pdf: 640,073 → 624,095 chars (2.5% removed)
✓ Saved to data\cleaned\annual_report_2024_cleaned.txt

✅ Cleaned and saved 1 documents


### Fixed Chunking

In [23]:
from  src.utils.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)

def get_splitter(strategy: str, chunk_size: int = 1500, chunk_overlap: int = 150):
    """
    Return a text splitter based on the specified strategy.
    
    Args:
        strategy: One of "recursive", "fixed", or "sentence"
        chunk_size: Maximum characters per chunk
        chunk_overlap: Overlap between consecutive chunks
        
    Returns:
        A LangChain text splitter instance
    """
    if strategy == "recursive": 
        return RecursiveCharacterTextSplitter(
                                            chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap,
                                            separators=["\n\n", "\n", ". ", ", ", " ", ""]
                                            )

    elif strategy == "fixed":
        return CharacterTextSplitter(
                                    chunk_size=chunk_size,
                                    chunk_overlap=chunk_overlap,
                                    separator=" "
                                    )

    elif strategy == "sentence":
        pass 

    else:
        raise ValueError("Unknown stratergy")
    



In [24]:
from pathlib import Path
from langchain_core.documents import Document

txt_path = Path("C:/Users/viraj/Zuu/Ledger_mind/data/cleaned/annual_report_2024_cleaned.txt")  # update this path
text = txt_path.read_text(encoding="utf-8")

document = [Document(page_content=text, metadata={"source": str(txt_path)})]
split_results = {}
strategy = "fixed"
splitter = get_splitter(strategy)
chunks = splitter.split_documents(document)

# Add strategy to metadata
for chunk in chunks:
    chunk.metadata["splitter"] = strategy

split_results[strategy] = chunks
print(f"✅ {strategy:10s}: {len(chunks):4d} chunks")

print(f"\nExample chunk ({strategy}):")
print(f"  Length: {len(split_results[strategy][0].page_content)} chars")
print(f"  Content: {split_results[strategy][0].page_content[:150]}...")
print(f"  Metadata: {split_results[strategy][0].metadata}")

✅ fixed     :  464 chunks

Example chunk (fixed):
  Length: 1490 chars
  Content: On Our Way
2024 ANNUAL REPORT
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who a...
  Metadata: {'source': 'C:\\Users\\viraj\\Zuu\\Ledger_mind\\data\\cleaned\\annual_report_2024_cleaned.txt', 'chunk_index': 0, 'splitter': 'fixed'}


In [33]:

chunks_file = output_dir / "chunks.jsonl"
with open(chunks_file, "w") as f:
    for i, chunk in enumerate(chunks):
        chunk_data = {"chunk_index": i, "content": chunk.page_content, "source": chunk.metadata["source"]}
        f.write(json.dumps(chunk_data) + "\n")

### Chroma DB

In [25]:
from langchain_chroma import Chroma

embeddings = get_text_embeddings(config)

from src.services.llm_services import load_config
config = load_config("src/config/config.yaml")

chroma_root = Path(config["artifacts_root"]) / "chroma"
chroma_root.mkdir(parents=True, exist_ok=True)

vectorstores = {}
strategy = "fixed"


collection_name = f"langchain_{strategy}"
persist_dir = str(chroma_root / collection_name)
    
print(f"Building collection: {collection_name}...")
    
vectorstore = Chroma.from_documents(
                                    documents=split_results[strategy],
                                    embedding=embeddings,
                                    collection_name=collection_name,
                                    persist_directory=persist_dir
                                    )
    
print(f"  ✅ Persisted to {persist_dir}")
print(f"  ✅ {len(split_results[strategy])} chunks embedded")

vectorstores[strategy] = vectorstore

print(f"\n✅ All collection built!")


ModuleNotFoundError: No module named 'langchain_huggingface'

In [ ]:


manifests_dir = Path(config["artifacts_root"]) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)

for strategy in strategies:
    manifest = {
        "collection_name": f"langchain_{strategy}",
        "framework": "langchain",
        "strategy": strategy,
        "embedding_model": config["text_emb_model"],
        "embedding_provider": config["text_emb_provider"],
        "normalize": config["normalize_embeddings"],
        "chunk_size": 800,
        "chunk_overlap": 150,
        "num_chunks": len(split_results[strategy]),
        "num_documents": len(documents),
        "created_at": datetime.now().isoformat(),
    }
    
    manifest_path = manifests_dir / f"langchain_{strategy}.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    
    print(f"✅ Manifest saved: {manifest_path.name}")


### QA Session

In [28]:
from src.services.qa_generator import QAPairGenerator

# Create dual-LLM configuration
qa_config = {
    "question_llm": {
        "llm_provider": "groq",
        "llm_model": "llama-3.1-8b-instant",
        "temperature": 0.7,  # Higher for diverse questions
        "max_tokens": 512,
        "request_timeout": 60
    },
    "answer_llm": {
        "llm_provider": "groq",
        "llm_model": "llama-3.1-8b-instant",
        "temperature": 0.2,  # Lower for factual answers
        "max_tokens": 384,
        "request_timeout": 60
    }
}

# Initialize QA generator
qa_generator = QAPairGenerator(
    config=qa_config,
    num_questions=10  # Generate 10 Q/A pairs per chunk
)

print("✅ QA Generator initialized")
print(f"   Question LLM: {qa_config['question_llm']['llm_model']} (temp={qa_config['question_llm']['temperature']})")
print(f"   Answer LLM: {qa_config['answer_llm']['llm_model']} (temp={qa_config['answer_llm']['temperature']})")
print(f"   Q/A pairs per chunk: 10")

✅ QA Generator initialized
   Question LLM: llama-3.1-8b-instant (temp=0.7)
   Answer LLM: llama-3.1-8b-instant (temp=0.2)
   Q/A pairs per chunk: 10


In [ ]:
# Test on first chunk
test_chunk = chunks[0]

print(f"Testing on chunk {test_chunk.metadata['chunk_index']}")
print(f"Chunk length: {len(test_chunk.page_content)} chars")
print(f"Content preview:\n{test_chunk.page_content[:300]}...\n")

print("Generating Q/A pairs...")
qa_pairs = qa_generator.generate_qa_pairs(test_chunk)

print(f"\n✅ Generated {len(qa_pairs)} Q/A pairs\n")
print("=" * 80)

# Display results
for i, pair in enumerate(qa_pairs, 1):
    print(f"\n{i}. Q: {pair['question']}")
    print(f"   A: {pair['answer']}")
    print(f"   ID: {pair['qa_pair_id']}")

print("\n" + "=" * 80)

Testing on chunk 0
Chunk length: 1490 chars
Content preview:
On Our Way
2024 ANNUAL REPORT
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our
mission to help people go anywhere and get anything and earn their way.
Movement is what we power. It’s our lifeblood. It runs...

Generating Q/A pairs...

✅ Generated 10 Q/A pairs


1. Q: What is the mission of Uber according to the company's description?
   A: We reimagine the way the world moves for the better. We are relentless about our mission to help people go anywhere and get anything and earn their way.
   ID: chunk_0_qa_1

2. Q: In what year did the fiscal year end for the annual report?
   A: December 31, 2024
   ID: chunk_0_qa_2

3. Q: According to the document, what is the lifeblood of Uber?
   A: Movement is the lifeblood of Uber.
   ID: chunk_0_qa_3

4. Q: How many places Uber aims to help people move around the world?
   A: Information not

In [29]:
chunks_to_process = chunks  # All 464 chunks (~4,640 Q/A pairs)

print(f"Processing {len(chunks_to_process)} chunks...")
print(f"Expected Q/A pairs: ~{len(chunks_to_process) * 10}\n")

# Generate Q/A pairs with progress tracking
all_qa_pairs = qa_generator.batch_generate(chunks_to_process, show_progress=True)

print(f"\n✅ Generation complete!")
print(f"   Total Q/A pairs: {len(all_qa_pairs)}")
print(f"   Chunks processed: {len(chunks_to_process)}")
print(f"   Avg pairs per chunk: {len(all_qa_pairs) / len(chunks_to_process):.1f}")

Processing 464 chunks...
Expected Q/A pairs: ~4640



Generating Q/A pairs:  42%|████▏     | 195/464 [54:11<1:03:09, 14.09s/it] 

Generating Q/A pairs:  55%|█████▍    | 253/464 [1:15:49<4:31:15, 77.14s/it] 

Generating Q/A pairs:  65%|██████▌   | 302/464 [1:29:13<1:21:43, 30.27s/it]


Failed on chunk 301: Connection error.


...


Failed on chunk 458: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km7gg27yfbq80632423s15z5` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499887, Requested 670. Please try again in 1m36.2496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Failed on chunk 459: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km7gg27yfbq80632423s15z5` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499887, Requested 566. Please try again in 1m18.2784s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Generating Q/A pairs: 100%|█████████▉| 462/464 [1:32:51<00:00,  7.42it/s]


Failed on chunk 460: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km7gg27yfbq80632423s15z5` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499886, Requested 571. Please try again in 1m18.9696s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Failed on chunk 461: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km7gg27yfbq80632423s15z5` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499885, Requested 590. Please try again in 1m22.08s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Generating Q/A pairs: 100%|██████████| 464/464 [1:32:51<00:00, 12.01s/it]


Failed on chunk 462: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km7gg27yfbq80632423s15z5` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499884, Requested 602. Please try again in 1m23.9808s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Failed on chunk 463: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km7gg27yfbq80632423s15z5` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499884, Requested 487. Please try again in 1m4.1088s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

⚠ Warning: 146 chunks failed: [301, 302, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 3

In [30]:
import json
from datetime import datetime

# Create output directory
output_dir = Path("data/qa_pairs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save to JSONL format (one Q/A pair per line)
output_file = output_dir / "uber_report_qa_pairs.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for pair in all_qa_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"✅ Saved {len(all_qa_pairs)} Q/A pairs to:")
print(f"   {output_file}")
print(f"   File size: {output_file.stat().st_size / 1024:.1f} KB")

# Save metadata
metadata = {
    "source_document": "annual_report_2024_cleaned.txt",
    "num_chunks_total": len(chunks),
    "num_chunks_processed": len(chunks_to_process),
    "num_qa_pairs": len(all_qa_pairs),
    "pairs_per_chunk": 10,
    "generated_at": datetime.now().isoformat(),
    "chunk_strategy": "fixed",
    "chunk_size": 1500,
    "chunk_overlap": 150,
    "llm_config": qa_config
}

metadata_file = output_dir / "uber_report_qa_pairs_metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Saved metadata to:")
print(f"   {metadata_file}")

✅ Saved 3180 Q/A pairs to:
   data\qa_pairs\uber_report_qa_pairs.jsonl
   File size: 1122.0 KB

✅ Saved metadata to:
   data\qa_pairs\uber_report_qa_pairs_metadata.json
